# 1-практика · Тобокелдик классификациясы (Risk Classification)

**Кыргыз Республикасынын Эсептөө палатасы · AI тренинги · 1-күн**

Бул — Jupyter дептери (notebook). Ал текст блокторунан жана код блокторунан (cell) турат.

**Кантип иштетүү керек:**
- Код блогун тандап, ▶ баскычын басыңыз (же **Shift + Enter**)
- Жыйынтык блоктун астында чыгат
- Блокторду **иретте, жогорудан ылдый** иштетиңиз

⚠️ **Эч нерсени бузуп алуу мүмкүн эмес.** Эң жаманы — баарын өчүрүп, кайра баштайбыз 🙂

## 0-кадам · Даярдык

Төмөнкү блок керектүү куралдарды орнотот. Бир мүнөтчө күтө туруңуз.

In [ ]:
%pip install -q pandas scikit-learn matplotlib

## 1-кадам · Салам, Python!

Биринчи ийгилигиңиз — ушул блокту иштетиңиз:

In [ ]:
print("Салам! Мен иштеп жатам ✓")
2 + 2

## 2-кадам · Маалыматтарды түзөбүз

Азыр **250 мекеменин** маалыматын түзөбүз. Ар бир мекеме боюнча:

| Мамыча | Мааниси |
|---|---|
| `аймак`, `тип` | мекеме кайда жана кандай |
| `бюджет_млн_сом` | жылдык бюджети (млн сом) |
| `мурунку_бузуулар` | өткөн аудитте табылган бузуулардын саны |
| `бир_булак_пайыз` | бир булактан сатып алуулардын үлүшү (%) |
| `кечигүү_күн` | отчетту канча күнгө кечиктирген |
| `четтөө_пайыз` | бюджетти аткаруудагы четтөө (%) |
| `тобокелдик` | **класс (label): Жогорку же Төмөн** |

⚠️ **МААНИЛҮҮ:** Маалымат толугу менен **синтетикалык** (ойдон чыгарылган, окшоштурулган).
Реалдуу мекемелерге окшоштук — жөн гана кокустук.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)   # seed — жыйынтык ар дайым бирдей болушу үчүн
n = 250

df = pd.DataFrame({
    "аймак": rng.choice(["Бишкек","Ош","Чүй","Ысык-Көл","Жалал-Абад","Нарын","Талас","Баткен"], n),
    "тип": rng.choice(["мектеп","оорукана","муниципалдык ишкана","мамлекеттик мекеме","айыл өкмөтү"], n),
    "бюджет_млн_сом": np.round(rng.uniform(5, 500, n), 1),
    "мурунку_бузуулар": rng.poisson(1.4, n).clip(0, 8),
    "бир_булак_пайыз": np.round(rng.beta(2, 4, n) * 90, 0),
    "кечигүү_күн": rng.poisson(8, n).clip(0, 60),
    "четтөө_пайыз": np.round(np.abs(rng.normal(8, 7, n)).clip(0, 30), 1),
})

# Жашыруун эреже (модель муну билбейт — өзү табышы керек!)
score = (1.15*(df["мурунку_бузуулар"] >= 3)
       + 0.85*(df["бир_булак_пайыз"] > 50)
       + 0.65*(df["четтөө_пайыз"] > 15)
       + 0.35*(df["кечигүү_күн"] > 20)
       + rng.normal(0, 0.42, n))
df["тобокелдик"] = np.where(score > 1.05, "Жогорку", "Төмөн")

print("Даяр! Мекемелердин саны:", len(df))

## 3-кадам · Маалыматка көз чаптыралы

`head(10)` — биринчи 10 сапты көрсөтөт (таблицанын «башы»).

In [ ]:
df.head(10)

Класстар кандай бөлүнгөн? Канча «Жогорку», канча «Төмөн»?

In [ ]:
import matplotlib.pyplot as plt

саноо = df["тобокелдик"].value_counts()
print(саноо)

саноо.plot(kind="bar", color=["#2C5F2D", "#C9A227"])
plt.title("Тобокелдик боюнча бөлүнүшү")
plt.ylabel("Мекемелердин саны")
plt.xticks(rotation=0)
plt.show()

## 4-кадам · Окутуу жана сынак (Train / Test)

Крокодил оюнун эстейли: сүрөттөрдүн **бир бөлүгүн сынакка каттык** — сиз аларды окутуу учурунда көргөн жоксуз.

Машина менен да так ошондой:
- **75%** — окутууга (training data)
- **25%** — сынакка **катып коёбуз** (test data)

Эмне үчүн? Модель «жаттап алганын» эмес, **чын эле үйрөнгөнүн** текшериш үчүн.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

белгилер = ["бюджет_млн_сом", "мурунку_бузуулар", "бир_булак_пайыз", "кечигүү_күн", "четтөө_пайыз"]
X = df[белгилер]
y = (df["тобокелдик"] == "Жогорку").astype(int)   # 1 = Жогорку, 0 = Төмөн

X_окутуу, X_сынак, y_окутуу, y_сынак = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

модель = DecisionTreeClassifier(max_depth=3, random_state=42)
модель.fit(X_окутуу, y_окутуу)

print("Окутууга берилди:", len(X_окутуу), "мекеме")
print("Сынакка катылды: ", len(X_сынак), "мекеме")

## 5-кадам · Модель ЭМНЕНИ үйрөндү? Дарагын карайлы

Биз **чечим дарагын (Decision Tree)** окуттук — суроолордон турган модель.

Ар бир төрт бурчтук — суроо. Жооп «ооба» болсо — солго, «жок» болсо — оңго.
Түс: **токойдой жашыл = Төмөн тобокелдикке жакын, саргыч = Жогоркуга жакын.**

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 8))
plot_tree(модель, feature_names=белгилер, class_names=["Төмөн", "Жогорку"],
          filled=True, rounded=True, fontsize=9, impurity=False)
plt.title("Модель тапкан эрежелер")
plt.show()

**Талкууга суроо:** Даракты окуп көрүңүз. Модель тапкан эрежелер сиздин аудитордук тажрыйбаңызга дал келеби?

Байкаңыз: бул эрежелерди **эч ким жазган жок** — модель аларды мисалдардан **өзү тапты**.

## 6-кадам · Божомол (Prediction)

Модель сынактагы (катылган!) мекемелерди көрөт. Биринчи 8ин салыштыралы:

In [ ]:
ыктымалдуулук = модель.predict_proba(X_сынак)[:, 1]
божомол = модель.predict(X_сынак)

жыйынтык = X_сынак.copy()
жыйынтык["ЧЫНДЫГЫ"] = np.where(y_сынак == 1, "Жогорку", "Төмөн")
жыйынтык["МОДЕЛДИН БОЖОМОЛУ"] = np.where(божомол == 1, "Жогорку", "Төмөн")
жыйынтык["Жогорку болуу ыктымалдыгы"] = (ыктымалдуулук * 100).round(0).astype(int).astype(str) + "%"

жыйынтык.head(8)

Бүдөмүк сүрөттү эстейли: модель да «100% ишенбейт» — ал **ыктымалдуулук (probability)** менен сүйлөйт.

## 7-кадам · Тактык (Accuracy)

Сынактагы мекемелердин канча пайызын туура таптык?

In [ ]:
from sklearn.metrics import accuracy_score

тактык = accuracy_score(y_сынак, божомол)
print(f"Моделдин тактыгы (accuracy): {тактык:.1%}")

## 🎉 Куттуктайбыз — сиз AI модель окуттуңуз!

**87% тактык.** Жакшы көрсөткүчпү?

Шашпаңыз... Кийинки дептерде бул санды **аудитордук көз караш** менен текшеребиз.
Сюрприз күтөт 🙂

**→ Ачыңыз: `2_confusion_matrix.ipynb`**